# Rohingya MMS Model Server (Colab T4 GPU -- browser or VS Code kernel connection both work)

Serves the two confirmed-real Rohingya models over HTTP so the local FastAPI backend (which has no GPU) can call them:
- **ASR**: `facebook/mms-1b-all` with the `rhg` adapter
- **TTS**: `facebook/mms-tts-rhg`

**Before running:**
1. Make sure this notebook is connected to a Colab T4 GPU runtime (via colab.research.google.com, or via VS Code's "Select Kernel -> Colab" picker -- both give you the real GPU).
2. Accept the model license/terms on the model pages if prompted the first time you load them: https://huggingface.co/facebook/mms-1b-all and https://huggingface.co/facebook/mms-tts-rhg
3. Get a Hugging Face access token (https://huggingface.co/settings/tokens, **Read** role is enough) and an ngrok authtoken (https://dashboard.ngrok.com/get-started/your-authtoken, free account is fine).
4. **Colab Secrets (the 🔑 key icon) only work when this notebook is open in the actual Colab website** -- they fail with a timeout if you're connected via VS Code's kernel picker. If you're on the Colab website, feel free to set `HF_TOKEN` and `NGROK_TOKEN` there (toggle "Notebook access" on). If you're on VS Code, or no secret is set, the cells below fall back to a hidden `getpass` prompt automatically.
5. **Never paste a real token as the *argument* to `getpass.getpass("...")`.** That argument is only the label shown above the input box -- running the cell pops up a separate hidden field below it, and that's where the secret goes, typed at runtime. A token pasted into the argument string gets saved in this file's plaintext on disk.
6. Run all cells top to bottom. The last cell prints a public `https://*.ngrok-free.app` URL -- paste that into `backend/.env` as `MODEL_SERVER_URL`.

**Important diagnostic**: the diagnostic cell below prints the raw ASR transcription of a real Rohingya clip it pulls automatically. Read that output carefully -- it tells us what script (Latin "Rohingyalish" vs Bangla-based) the model actually produces, which the glossary matching in the backend currently assumes is Latin. If it comes out differently, tell Claude so `backend/app/glossary.py` can be adjusted.

In [ ]:
!pip install -q transformers accelerate torch torchaudio librosa soundfile fastapi uvicorn pyngrok nest_asyncio python-multipart huggingface_hub

In [ ]:
import getpass

from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if not hf_token:
    hf_token = getpass.getpass(
        "No HF_TOKEN Colab secret available here -- paste your Hugging Face access token: "
    )

login(token=hf_token)

In [ ]:
import torch
from transformers import Wav2Vec2ForCTC, AutoProcessor, VitsModel, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

ASR_MODEL_ID = "facebook/mms-1b-all"
TTS_MODEL_ID = "facebook/mms-tts-rhg"

print("loading ASR processor/model (rhg adapter)...")
asr_processor = AutoProcessor.from_pretrained(ASR_MODEL_ID, target_lang="rhg")
asr_model = Wav2Vec2ForCTC.from_pretrained(ASR_MODEL_ID, target_lang="rhg", ignore_mismatched_sizes=True).to(device)
asr_model.load_adapter("rhg")
asr_model.eval()

print("loading TTS model...")
tts_tokenizer = AutoTokenizer.from_pretrained(TTS_MODEL_ID)
tts_model = VitsModel.from_pretrained(TTS_MODEL_ID).to(device)
tts_model.eval()

print("models loaded.")

device: cuda
loading ASR processor/model (rhg adapter)...


preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.04k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  145MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

models loaded.


In [ ]:
import numpy as np
import librosa

SAMPLE_RATE = 16000


def run_asr(audio_path_or_array, sr=None):
    if isinstance(audio_path_or_array, (str,)):
        audio, _ = librosa.load(audio_path_or_array, sr=SAMPLE_RATE, mono=True)
    else:
        audio = audio_path_or_array
        if sr is not None and sr != SAMPLE_RATE:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)
    inputs = asr_processor(audio, sampling_rate=SAMPLE_RATE, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = asr_model(**inputs).logits
    ids = torch.argmax(logits, dim=-1)[0]
    return asr_processor.decode(ids)


def run_tts(text: str) -> tuple[np.ndarray, int]:
    inputs = tts_tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = tts_model(**inputs).waveform
    waveform = output.squeeze().cpu().numpy()
    return waveform, tts_model.config.sampling_rate

### Diagnostic: pull one real Rohingya clip and sanity-check both legs
The dataset's shards are ~800MB tar files, so we don't download a whole one -- we open it as an HTTP stream and read just far enough to grab the first audio entry, then close the connection. Works the same whether this kernel is the Colab website or a Colab GPU runtime attached through VS Code (unlike `google.colab.files.upload()` / `google.colab.userdata`, which need the actual Colab web page and fail with a timeout otherwise).

In [ ]:
import tarfile
import requests

SHARD_URL = "https://huggingface.co/datasets/freococo/rohingya_asr_audio/resolve/main/train/rohingya-00000.tar"

resp = requests.get(SHARD_URL, stream=True, headers={"Authorization": f"Bearer {hf_token}"} if hf_token else {})
resp.raise_for_status()
sample_bytes, sample_name = None, None
with tarfile.open(fileobj=resp.raw, mode="r|") as tar:
    for member in tar:
        if member.isfile() and member.size > 0:
            f = tar.extractfile(member)
            if f:
                sample_bytes = f.read()
                sample_name = member.name
                break
resp.close()

assert sample_bytes, "couldn't find an audio entry in the first part of the shard"
print(f"pulled {sample_name} ({len(sample_bytes)} bytes) without downloading the full 800MB shard")

sample_path = "sample_rhg.mp3"
with open(sample_path, "wb") as out:
    out.write(sample_bytes)

transcription = run_asr(sample_path)
print("RAW ASR OUTPUT (read this carefully, report back the script/characters you see):")
print(repr(transcription))

pulled 000001.mp3 (104532 bytes) without downloading the full 800MB shard
RAW ASR OUTPUT (read this carefully, report back the script/characters you see):
'gásof amérika wacíngthon'


In [ ]:
import soundfile as sf
from IPython.display import Audio

test_text = transcription if transcription.strip() else "test"
wav, sr = run_tts(test_text)
sf.write("tts_test.wav", wav, sr)
Audio("tts_test.wav")

### Model server: FastAPI + ngrok tunnel
Leave this cell running -- it's the server the local backend will call. Stopping it (or the Colab runtime disconnecting) kills the tunnel.

In [ ]:
import io
import threading

import nest_asyncio
import soundfile as sf
import uvicorn
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import Response
from pydantic import BaseModel
from pyngrok import ngrok, conf

nest_asyncio.apply()

app = FastAPI()


@app.get("/health")
def health():
    return {"status": "ok", "device": device}


@app.post("/asr")
async def asr_endpoint(file: UploadFile = File(...)):
    raw = await file.read()
    audio, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    text = run_asr(audio, sr=sr)
    return {"text": text}


class TTSRequest(BaseModel):
    text: str


@app.post("/tts")
def tts_endpoint(req: TTSRequest):
    waveform, sr = run_tts(req.text)
    buf = io.BytesIO()
    sf.write(buf, waveform, sr, format="WAV")
    return Response(content=buf.getvalue(), media_type="audio/wav")


try:
    from google.colab import userdata
    ngrok_token = userdata.get("NGROK_TOKEN")
except Exception:
    ngrok_token = None

if not ngrok_token:
    ngrok_token = getpass.getpass(
        "No NGROK_TOKEN Colab secret available here -- paste your ngrok authtoken: "
    )

conf.get_default().auth_token = ngrok_token
public_url = ngrok.connect(8000)
print("\nMODEL SERVER URL (paste into backend/.env as MODEL_SERVER_URL):")
print(public_url)


def _run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")


threading.Thread(target=_run, daemon=True).start()

                                                                                                    
MODEL SERVER URL (paste into backend/.env as MODEL_SERVER_URL):
NgrokTunnel: "https://conceded-seventeen-bring.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [571]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [ ]:
import sys, os
print("python:", sys.version)
print("is colab module present:", "google.colab" in sys.modules)
try:
    import google.colab
    print("google.colab importable: True")
except Exception as e:
    print("google.colab importable: False ->", e)

import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)


python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
is colab module present: True
google.colab importable: True
name, memory.total [MiB]
Tesla T4, 15360 MiB



In [ ]:
#from pyngrok import ngrok
#ngrok.kill()

INFO:     142.172.82.80:0 - "POST /tts HTTP/1.1" 200 OK
INFO:     142.172.82.80:0 - "POST /tts HTTP/1.1" 200 OK
INFO:     142.172.82.80:0 - "POST /tts HTTP/1.1" 200 OK
INFO:     142.172.82.80:0 - "POST /tts HTTP/1.1" 200 OK
INFO:     142.172.82.80:0 - "POST /asr HTTP/1.1" 200 OK
INFO:     142.172.82.80:0 - "POST /tts HTTP/1.1" 200 OK
INFO:     142.172.82.80:0 - "POST /asr HTTP/1.1" 200 OK
INFO:     142.172.82.80:0 - "POST /tts HTTP/1.1" 200 OK
INFO:     142.172.82.80:0 - "POST /tts HTTP/1.1" 200 OK
INFO:     142.172.82.80:0 - "POST /asr HTTP/1.1" 200 OK
INFO:     142.172.82.80:0 - "POST /tts HTTP/1.1" 200 OK
INFO:     142.172.82.80:0 - "POST /tts HTTP/1.1" 200 OK
INFO:     142.172.82.80:0 - "POST /asr HTTP/1.1" 200 OK
INFO:     142.172.82.80:0 - "POST /tts HTTP/1.1" 200 OK
INFO:     142.172.82.80:0 - "POST /tts HTTP/1.1" 200 OK
INFO:     142.172.82.80:0 - "POST /asr HTTP/1.1" 200 OK
INFO:     142.172.82.80:0 - "POST /tts HTTP/1.1" 200 OK
INFO:     142.172.82.80:0 - "POST /tts HTTP/1.1"